# 03_Gold_Medallion

Esta notebook lee el archivo Parquet limpio generado en Silver, crea un modelo estrella con tabla de hechos y dimensiones, y exporta los resultados en formato Excel o CSV según la variable `salida`: Excel para gerencia (fácilmente legible) o CSV para BI (óptimo almacenamiento y lectura).

In [ ]:
#!pip install pandas requests pyarrow openpyxl

### TIPO DE SALIDA
#### "gerencia" para Excel separado y consolidado
#### "bi" para CSV separados

In [ ]:
# Definir tipo de salida
salida = "gerencia"

In [ ]:
from pathlib import Path
import pandas as pd

In [ ]:
SILVER_PATH = Path('data/silver/superstore_clean.parquet')
GOLD_DIR = Path('data/gold')
GOLD_DIR.mkdir(parents=True, exist_ok=True)

try:
    df = pd.read_parquet(SILVER_PATH)
    print('Loaded silver parquet with shape:', df.shape)
except Exception as exc:
    print('Error loading silver parquet:')
    print(str(exc))

In [ ]:
try:
    # Crear dimensión de fecha
    date_dim = (
        df[['order_date']]
        .drop_duplicates()
        .reset_index(drop=True)
        .assign(
            date_key=lambda d: d['order_date'].dt.strftime('%Y%m%d').astype(int),
            year=lambda d: d['order_date'].dt.year,
            quarter=lambda d: d['order_date'].dt.quarter,
            month=lambda d: d['order_date'].dt.month,
            month_name=lambda d: d['order_date'].dt.month_name(),
            day=lambda d: d['order_date'].dt.day,
            weekday=lambda d: d['order_date'].dt.day_name()
        )
        .loc[:, ['date_key', 'order_date', 'year', 'quarter', 'month', 'month_name', 'day', 'weekday']]
    )
    date_dim = date_dim.sort_values('order_date').reset_index(drop=True)
    print('Dimensión de fecha creada con shape:', date_dim.shape)
except Exception as exc:
    print('Error creando dimensión de fecha:')
    print(str(exc))

In [ ]:
try:
    # Crear dimensión de cliente
    cust_cols = ['customer_id', 'customer_name', 'segment', 'country', 'city', 'state', 'postal_code', 'region']
    customer_dim = df[cust_cols].drop_duplicates().reset_index(drop=True)
    customer_dim['customer_key'] = customer_dim.index + 1
    customer_dim = customer_dim[['customer_key'] + cust_cols]
    print('Dimensión de cliente creada con shape:', customer_dim.shape)
except Exception as exc:
    print('Error creando dimensión de cliente:')
    print(str(exc))

In [ ]:
try:
    # Crear dimensión de producto
    prod_cols = ['product_id', 'product_name', 'category', 'sub_category']
    product_dim = df[prod_cols].drop_duplicates().reset_index(drop=True)
    product_dim['product_key'] = product_dim.index + 1
    product_dim = product_dim[['product_key'] + prod_cols]
    print('Dimensión de producto creada con shape:', product_dim.shape)
except Exception as exc:
    print('Error creando dimensión de producto:')
    print(str(exc))

In [ ]:
try:
    # Crear dimensión de envío
    ship_cols = ['ship_mode']
    ship_dim = df[ship_cols].drop_duplicates().reset_index(drop=True)
    ship_dim['ship_key'] = ship_dim.index + 1
    ship_dim = ship_dim[['ship_key', 'ship_mode']]
    print('Dimensión de envío creada con shape:', ship_dim.shape)
except Exception as exc:
    print('Error creando dimensión de envío:')
    print(str(exc))

In [ ]:
try:
    # Construir tabla de hechos usando claves surrogadas
    fact_df = df.merge(date_dim[['order_date', 'date_key']], on='order_date', how='left')
    fact_df = fact_df.merge(customer_dim, on=cust_cols, how='left')
    fact_df = fact_df.merge(product_dim, on=prod_cols, how='left')
    fact_df = fact_df.merge(ship_dim, on=ship_cols, how='left')

    fact_table = fact_df[
        [
            'order_id',
            'date_key',
            'customer_key',
            'product_key',
            'ship_key',
            'sales',
            'quantity',
            'discount',
            'profit',
            'profit_margin',
            'order_date',
            'ship_date',
            'region',
            'state'
        ]
    ].drop_duplicates().reset_index(drop=True)
    fact_table['fact_key'] = fact_table.index + 1
    fact_table = fact_table[
        ['fact_key', 'order_id', 'date_key', 'customer_key', 'product_key', 'ship_key', 'sales', 'quantity', 'discount', 'profit', 'profit_margin', 'order_date', 'ship_date', 'region', 'state']
    ]
    print('Tabla de hechos creada con shape:', fact_table.shape)
except Exception as exc:
    print('Error creando tabla de hechos:')
    print(str(exc))

In [ ]:
# Definir rutas de archivos según tipo de salida
if salida == "gerencia":
    date_path = GOLD_DIR / 'dim_date.xlsx'
    customer_path = GOLD_DIR / 'dim_customer.xlsx'
    product_path = GOLD_DIR / 'dim_product.xlsx'
    ship_path = GOLD_DIR / 'dim_ship.xlsx'
    fact_path = GOLD_DIR / 'fact_sales.xlsx'
    all_path = GOLD_DIR / 'superstore_star_schema.xlsx'
elif salida == "bi":
    date_path = GOLD_DIR / 'dim_date.csv'
    customer_path = GOLD_DIR / 'dim_customer.csv'
    product_path = GOLD_DIR / 'dim_product.csv'
    ship_path = GOLD_DIR / 'dim_ship.csv'
    fact_path = GOLD_DIR / 'fact_sales.csv'
    all_path = None  # No consolidado para BI

In [ ]:
try:
    # Exportar dimensión de fecha
    if salida == "gerencia":
        date_dim.to_excel(date_path, index=False)
    elif salida == "bi":
        date_dim.to_csv(date_path, index=False)
    print('Dimensión de fecha exportada a:', date_path.name)
except Exception as exc:
    print('Error exportando dimensión de fecha:')
    print(str(exc))

In [ ]:
try:
    # Exportar dimensión de cliente
    if salida == "gerencia":
        customer_dim.to_excel(customer_path, index=False)
    elif salida == "bi":
        customer_dim.to_csv(customer_path, index=False)
    print('Dimensión de cliente exportada a:', customer_path.name)
except Exception as exc:
    print('Error exportando dimensión de cliente:')
    print(str(exc))

In [ ]:
try:
    # Exportar dimensión de producto
    if salida == "gerencia":
        product_dim.to_excel(product_path, index=False)
    elif salida == "bi":
        product_dim.to_csv(product_path, index=False)
    print('Dimensión de producto exportada a:', product_path.name)
except Exception as exc:
    print('Error exportando dimensión de producto:')
    print(str(exc))

In [ ]:
try:
    # Exportar dimensión de envío
    if salida == "gerencia":
        ship_dim.to_excel(ship_path, index=False)
    elif salida == "bi":
        ship_dim.to_csv(ship_path, index=False)
    print('Dimensión de envío exportada a:', ship_path.name)
except Exception as exc:
    print('Error exportando dimensión de envío:')
    print(str(exc))

In [ ]:
try:
    # Exportar tabla de hechos
    if salida == "gerencia":
        fact_table.to_excel(fact_path, index=False)
    elif salida == "bi":
        fact_table.to_csv(fact_path, index=False)
    print('Tabla de hechos exportada a:', fact_path.name)
except Exception as exc:
    print('Error exportando tabla de hechos:')
    print(str(exc))

In [ ]:
if salida == "gerencia":
    try:
        # Exportar esquema estrella consolidado a Excel
        with pd.ExcelWriter(all_path, engine='openpyxl') as writer:
            date_dim.to_excel(writer, sheet_name='DIM_Date', index=False)
            customer_dim.to_excel(writer, sheet_name='DIM_Customer', index=False)
            product_dim.to_excel(writer, sheet_name='DIM_Product', index=False)
            ship_dim.to_excel(writer, sheet_name='DIM_Ship', index=False)
            fact_table.to_excel(writer, sheet_name='FACT_Sales', index=False)
        print('Esquema estrella consolidado exportado a:', all_path.name)
    except Exception as exc:
        print('Error exportando esquema consolidado:')
        print(str(exc))

In [ ]:
print(f'Archivos {"Excel" if salida == "gerencia" else "CSV"} de dimensiones y hechos guardados en', GOLD_DIR.resolve())
print('Archivos del esquema Gold:')
print('-', date_path.name)
print('-', customer_path.name)
print('-', product_path.name)
print('-', ship_path.name)
print('-', fact_path.name)
if salida == "gerencia":
    print('-', all_path.name)